In [2]:
# 提取CAIL单标签标准数据集
import json
import random
from tqdm import tqdm
random.seed(2023)

def digit_to_chinese(number):
    chinese_digits = {
        '0': '零',
        '1': '一',
        '2': '二',
        '3': '三',
        '4': '四',
        '5': '五',
        '6': '六',
        '7': '七',
        '8': '八',
        '9': '九',
    }

    chinese_units = ['', '十', '百', '千', '万']

    number_str = str(number)
    result = ""
    length = len(number_str)
    zero_flag = False

    for i, digit in enumerate(number_str):
        if digit == '0':
            zero_flag = True
            if i == length - 1:
                result += chinese_digits[digit]
            continue
        else:
            if zero_flag:
                result += chinese_digits['0']
                zero_flag = False
            result += chinese_digits[digit] + chinese_units[length - i - 1]

    return result

path = "/root/data1/liang/self-correct-retriever/data/train_data/ljp_split/data_test.json"
write_path = "/root/data1/liang/self-correct-retriever/data/knowledge_base/single_test_ljp_temp.json"
with open(path,"r") as f:
    lines = f.readlines()
    result = []
    random.shuffle(lines)
    for line in tqdm(lines[:1000]):
        data = json.loads(line)
        origin = data["fact"]
        article = data["meta"]["relevant_articles"]
        charge = data["meta"]["accusation"]
        penalty = data["meta"]["term_of_imprisonment"]["imprisonment"]
        if len(article)==1 and len(charge)==1:
            result.append({"fact":origin, "article":digit_to_chinese(article[0]), "charge":charge[0].replace("[","").replace("]",""), "penalty":penalty})
with open(write_path, "w") as f:
    random.shuffle(result)
    for dic in result:
        json.dump(dic,f,ensure_ascii=False)
        f.write("\n")


100%|██████████| 1000/1000 [00:00<00:00, 119021.11it/s]


In [3]:
# 依据测试集的标签筛选训练集
import pandas as pd
import json
import random
random.seed(2023)
with open("/root/data1/liang/self-correct-retriever/data/knowledge_base/single_test_ljp_temp.json","r") as f:
    data_all = []
    lines = f.readlines()
    random.shuffle(lines)
    for line in lines:
        line = json.loads(line)
        data_all.append(line)
original_dataset = pd.DataFrame(data_all)
accusation_counts = original_dataset["charge"].apply(lambda x: x).value_counts()
print(accusation_counts)

keys_with_values_less_than_5 = [key for key, value in accusation_counts.items() if value > 5]

print(len(keys_with_values_less_than_5))

filtered_data = original_dataset[original_dataset['charge'].isin(keys_with_values_less_than_5)]
print(len(filtered_data))
data_all=[]
for i in range(len(filtered_data)):
    data_all.append(filtered_data.iloc[i].to_dict())
with open("/root/data1/liang/self-correct-retriever/data/knowledge_base/single_test_ljp.json", "w") as f:
    random.shuffle(data_all)
    for dic in data_all:
        json.dump(dic,f,ensure_ascii=False)
        f.write("\n")


charge
盗窃              51
故意伤害            30
诈骗              28
合同诈骗            18
生产、销售有毒、有害食品    17
                ..
诬告陷害             1
制造、贩卖、传播淫秽物品     1
传播性病             1
行贿               1
伪证               1
Name: count, Length: 120, dtype: int64
56
683


In [4]:
accusation_counts = filtered_data["charge"].apply(lambda x: x).value_counts()
print(accusation_counts)

charge
盗窃                           51
故意伤害                         30
诈骗                           28
合同诈骗                         18
生产、销售有毒、有害食品                 17
危险驾驶                         17
强奸                           17
走私、贩卖、运输、制造毒品                17
非法制造、买卖、运输、邮寄、储存枪支、弹药、爆炸物    17
寻衅滋事                         15
交通肇事                         15
非法持有、私藏枪支、弹药                 14
妨害公务                         13
非国家工作人员受贿                    13
骗取贷款、票据承兑、金融票证               13
假冒注册商标                       13
敲诈勒索                         13
窝藏、包庇                        13
非法占用农用地                      13
生产、销售假药                      12
非法种植毒品原植物                    12
销售假冒注册商标的商品                  12
聚众斗殴                         11
非法吸收公众存款                     11
开设赌场                         11
故意杀人                         11
挪用公款                         11
污染环境                         11
放火                           11
非法行医                         10
拒不执行判决、裁定                    10
玩

In [5]:
# 构建标签平衡的新数据集
import pandas as pd
import json
import random
# 假设您有一个原始数据集 original_dataset
# 示例数据格式如下：
with open("/root/data1/liang/self-correct-retriever/data/knowledge_base/single_train_ljp.json","r") as f:
    data_all = []
    lines = f.readlines()
    for line in lines:
        line = json.loads(line)
        data_all.append(line)
original_dataset = pd.DataFrame(data_all)
original_dataset = original_dataset[original_dataset['charge'].isin(keys_with_values_less_than_5)]
# 检查每个罪名的数量
accusation_counts = original_dataset["charge"].apply(lambda x: x).value_counts()

# 设置每个罪名的目标数量（最多1000条）
target_count = 100

# 创建一个空的DataFrame来存储平衡后的数据
balanced_dataset = pd.DataFrame(columns=original_dataset.columns)

# 对每个罪名进行处理，确保数量不超过目标数量
for accusation, count in accusation_counts.items():
    subset = original_dataset[original_dataset["charge"].apply(lambda x: x) == accusation]
    if count > target_count:
        subset = subset.sample(target_count)  # 如果数量超过目标数量，随机采样一部分数据
    balanced_dataset = pd.concat([balanced_dataset, subset])
print(len(balanced_dataset))
print(balanced_dataset["charge"].apply(lambda x: x).value_counts())
data_all=[]
for i in range(len(balanced_dataset)):
    data_all.append(balanced_dataset.iloc[i].to_dict())
# 现在 balanced_dataset 就是平衡后的数据集，每个罪名不超过1000条
write_path = "/root/data1/liang/self-correct-retriever/data/knowledge_base/balanced_train_ljp.json"
with open(write_path, "w") as f:
    random.shuffle(data_all)
    for dic in data_all:
        json.dump(dic,f,ensure_ascii=False)
        f.write("\n")
print(len(data_all))   

5504
charge
生产、销售不符合安全标准的食品              100
生产、销售有毒、有害食品                 100
非法狩猎                         100
伪造公司、企业、事业单位、人民团体印章          100
赌博                           100
合同诈骗                         100
非国家工作人员受贿                    100
伪造、变造、买卖国家机关公文、证件、印章         100
开设赌场                         100
危险驾驶                         100
聚众斗殴                         100
故意伤害                         100
非法侵入住宅                       100
生产、销售伪劣产品                    100
掩饰、隐瞒犯罪所得、犯罪所得收益             100
非法持有、私藏枪支、弹药                 100
非法拘禁                         100
滥用职权                         100
信用卡诈骗                        100
寻衅滋事                         100
妨害公务                         100
抢夺                           100
破坏广播电视设施、公用电信设施              100
猥亵儿童                         100
盗窃                           100
抢劫                           100
走私、贩卖、运输、制造毒品                100
挪用公款                         100
故意毁坏财物                       100
敲诈勒索                         10